# 04 (LOCAL) — LLM Diagnostic Layer, run on your laptop

Runs the full diagnostic pipeline **locally** — no GPU, no Kaggle, no full dataset download.
It uses your local `models/unet_dropout.pt`, fetches only the JSONs + a **class-stratified sample**
of validation images from Hugging Face (a few MB), and calls **Gemini** (free tier) for each diagnosis.

**Sample size:** default 300, class-stratified — statistically solid, within Gemini's daily free cap,
and enough to report per-class diagnostic quality. Resumable: re-run if you hit a rate limit and it continues.

**Prereqs (local .venv):** `pip install google-generativeai segmentation-models-pytorch albumentations opencv-python huggingface_hub torch`
**API key:** set `GEMINI_API_KEY` as an environment variable before launching Jupyter/VS Code, or set it in cell 1.

## 1. Config + API key

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
assert os.path.isdir('src'), f'expected repo root, got {os.getcwd()}'
sys.path.insert(0, os.getcwd())

from dotenv import load_dotenv
load_dotenv()
assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY not found in .env at repo root'

PROVIDER   = 'gemini'
LLM_MODEL  = 'gemini-3.5-flash-lite'
CKPT       = 'models/unet_dropout.pt'
PER_CLASS  = 10
MC_PASSES  = 20
OUT_DIR    = 'results/llm'
os.makedirs(OUT_DIR, exist_ok=True)
print('cwd:', os.getcwd(), '| src:', os.path.isdir('src'),
      '| key:', bool(os.environ.get('GEMINI_API_KEY')), '| ckpt:', os.path.exists(CKPT))

cwd: c:\Users\HP PC\Projects\SteelDefectX | src found: True
key set: True | ckpt exists: True


## 2. Fetch ONLY the JSONs + a class-stratified image sample (a few MB)

In [ ]:
import json, random
from collections import defaultdict
from huggingface_hub import hf_hub_download
REPO = 'Zhaosxian/SteelDefectX'

def get_json(fn):
    p = hf_hub_download(repo_id=REPO, filename=fn, repo_type='dataset')
    return json.load(open(p, encoding='utf-8'))

t1 = get_json('class_descriptions.json')
val_text = get_json('val-text.json')
print('classes:', len(t1), '| val entries:', len(val_text))

random.seed(0)
by_cls = defaultdict(list)
for e in val_text:
    by_cls[e['class_name']].append(e)
sample = []
for c, items in by_cls.items():
    random.shuffle(items)
    sample += items[:PER_CLASS]
random.shuffle(sample)
print(f'stratified sample: {len(sample)} images, up to {PER_CLASS} per class across {len(by_cls)} classes')

def fetch_val_image(image_name):
    return hf_hub_download(repo_id=REPO, filename=f'val/{image_name}', repo_type='dataset')

classes: 25 | val entries: 2324
stratified sample: 250 images, up to 10 per class across 25 classes


## 3. Load the local dropout-U-Net

In [ ]:
import numpy as np, cv2, torch
from src.segmentation.model import build_model, enable_mc_dropout
from src.llm.attributes import extract_attributes
from src.llm.diagnose import diagnose
from src.llm.evaluate_llm import evaluate_batch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = build_model('unet', 'resnet34', None, dropout=0.2)
model.load_state_dict(torch.load(CKPT, map_location=device)); model.to(device).eval()
print('model loaded on', device)

MEAN = np.array([0.485,0.456,0.406])[None,:,None,None]
STD  = np.array([0.229,0.224,0.225])[None,:,None,None]

def predict_with_conf(img_path, mc=MC_PASSES):
    g = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE); g = cv2.resize(g, (256,256))
    x = np.stack([g,g,g],0)[None].astype(np.float32)/255.0
    x = torch.tensor((x-MEAN)/STD, dtype=torch.float32).to(device)
    enable_mc_dropout(model)
    with torch.no_grad():
        ps = torch.stack([torch.sigmoid(model(x)) for _ in range(mc)],0)
    prob = ps.mean(0)[0,0].cpu().numpy(); unc = ps.std(0)[0,0].cpu().numpy()
    conf = float(prob[prob>0.5].mean()) if (prob>0.5).any() else float(1-prob.mean())
    return g, prob, unc, conf

model loaded on cpu


## 4. Run the pipeline over the sample (RESUMABLE)

Saves each diagnosis as it goes. If you hit a Gemini rate limit, just re-run this cell — it skips
images already done and continues.

In [ ]:
import time
PROG = f'{OUT_DIR}/diagnoses.jsonl'
done = set()
if os.path.exists(PROG):
    for line in open(PROG):
        try: done.add(json.loads(line)['image_name'])
        except Exception: pass
print('already done:', len(done))

def run_one(nm, cl):
    g, prob, unc, conf = predict_with_conf(fetch_val_image(nm))
    attrs = extract_attributes(g, prob)
    diag = diagnose(cl, t1[cl], attrs, confidence=conf, uncertainty=float(unc.mean()),
                    provider=PROVIDER, model=LLM_MODEL)
    return {'image_name': nm, 'class_name': cl, 'confidence': round(conf,3),
            'attributes': {k:v for k,v in attrs.items() if k!='_raw'}, 'diag': diag}

fout = open(PROG, 'a'); errors = 0
for i, e in enumerate(sample):
    nm, cl = e['image_name'], e['class_name']
    if nm in done:
        continue
    try:
        rec = run_one(nm, cl)
        fout.write(json.dumps(rec)+'\n'); fout.flush()
        if (i+1) % 20 == 0: print(f'{i+1}/{len(sample)} done')
    except Exception as ex:
        errors += 1; msg = str(ex)
        if 'rate' in msg.lower() or 'quota' in msg.lower() or '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
            print(f'rate/quota limit at {i} — sleeping 60s then retrying once...')
            time.sleep(60)
            try:
                rec = run_one(nm, cl); fout.write(json.dumps(rec)+'\n'); fout.flush()
            except Exception:
                print('   still limited — interrupt & re-run this cell next session to resume.')
            continue
        print('skip', nm, msg[:120]); time.sleep(1)
fout.close(); print('errors:', errors)

already done: 0


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


val/rs_115.jpg: reconstructing file:   0%|          |  0.00B / 10.1kB            

val/rs_115.jpg: downloading bytes:           |  0.00B            

c:\Users\HP PC\Projects\SteelDefectX\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP PC\.cache\huggingface\hub\datasets--Zhaosxian--SteelDefectX. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


...

val/Ots_88.jpg: reconstructing file:   0%|          |  0.00B / 6.91kB            

val/Ots_88.jpg: downloading bytes:           |  0.00B            

val/cg_111.jpg: reconstructing file:   0%|          |  0.00B / 3.37kB            

val/cg_111.jpg: downloading bytes:           |  0.00B            

errors: 0


## 5. Aggregate metrics: overall + per-class

In [ ]:
from src.llm.evaluate_llm import check_structural_validity, check_grounding, check_faithfulness
recs = [json.loads(l) for l in open(PROG)]
print('total diagnoses:', len(recs))

batch = [{'diag': r['diag'], 't1_cause': t1[r['class_name']], 'attributes': r['attributes']} for r in recs]
overall = evaluate_batch(batch)
print('\nOVERALL:', json.dumps(overall, indent=2))

pc = defaultdict(lambda: {'n':0,'valid':0,'ground':0,'faith':0})
for r in recs:
    c = r['class_name']; d = r['diag']; a = r['attributes']
    pc[c]['n'] += 1
    pc[c]['valid']  += int(check_structural_validity(d))
    pc[c]['ground'] += int(check_grounding(d, t1[c]))
    pc[c]['faith']  += int(check_faithfulness(d, a))
print('\nPER-CLASS (validity / grounding / faithfulness %):')
for c in sorted(pc):
    s = pc[c]; n = s['n']
    print(f"  {c:32s} n={n:3d}  val={100*s['valid']/n:5.1f}  grd={100*s['ground']/n:5.1f}  fth={100*s['faith']/n:5.1f}")

json.dump({'overall': overall, 'per_class': {c: dict(v) for c,v in pc.items()}},
          open(f'{OUT_DIR}/eval_metrics.json','w'), indent=2)
print('\nsaved -> results/llm/eval_metrics.json')

total diagnoses: 250

OVERALL: {
  "n": 250,
  "structural_validity_pct": 100.0,
  "grounding_rate_pct": 100.0,
  "faithfulness_pct": 100.0
}

PER-CLASS (validity / grounding / faithfulness %):
  Bright scratch                   n= 10  val=100.0  grd=100.0  fth=100.0
  Crazing                          n= 10  val=100.0  grd=100.0  fth=100.0
  Crease                           n= 10  val=100.0  grd=100.0  fth=100.0
  Crescent gap                     n= 10  val=100.0  grd=100.0  fth=100.0
  Dark scratches                   n= 10  val=100.0  grd=100.0  fth=100.0
  Finishing roll printing          n= 10  val=100.0  grd=100.0  fth=100.0
  Inclusion                        n= 10  val=100.0  grd=100.0  fth=100.0
  Iron scale compression           n= 10  val=100.0  grd=100.0  fth=100.0
  Iron sheet ash                   n= 10  val=100.0  grd=100.0  fth=100.0
  Oil spot                         n= 10  val=100.0  grd=100.0  fth=100.0
  Oxide scale of plate system      n= 10  val=100.0  grd=100.0  ft

## 6. Show a few example diagnostic notes (for the report / demo)

In [ ]:
for r in recs[:5]:
    d = r['diag']
    print(f"=== {r['image_name']} [{r['class_name']}]  conf={r['confidence']} ===")
    print('  attributes:', {k:r['attributes'][k] for k in ('Shape','Scale','Polarity','Saliency') if k in r['attributes']})
    print('  cause   :', d.get('likely_cause'))
    print('  severity:', d.get('severity'))
    print('  action  :', d.get('recommended_action'))
    print('  summary :', d.get('summary'))
    print()

=== ds_220.jpg [Dark scratches]  conf=0.973 ===
  attributes: {'Shape': 'linear', 'Scale': 'small', 'Polarity': 'dark', 'Saliency': 'medium'}
  cause   : Mechanical abrasion or roller damage resulting in a vertical dark linear scratch on the steel surface.
  severity: moderate
  action  : Inspect upstream roller alignment and handling equipment for surface contact issues; hold coil for review.
  summary : A moderate vertical dark scratch was detected at the center, indicating localized mechanical abrasion from roller contact.

=== rs_115.jpg [Rolled in scale]  conf=0.9 ===
  attributes: {'Shape': 'fragmented', 'Scale': 'small', 'Polarity': 'dark', 'Saliency': 'medium'}
  cause   : Scale material has been pressed and rolled directly into the base metal, resulting in dark, fragmented particles embedded across the surface.
  severity: moderate
  action  : Inspect the descaling and roughing stages for operational failure; place a temporary hold on the affected coil for surface conditioning